In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["MKL_CBWR"] = "COMPATIBLE"
os.environ["ATEN_CPU_CAPABILITY"] = "avx2"
os.environ["OMP_NUM_THREADS"] = os.environ["MKL_NUM_THREADS"] = os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["NPY_DISABLE_CPU_FEATURES"] = "AVX512F AVX512CD AVX512_KNL AVX512_KNM AVX512_SKX AVX512_CLX AVX512_CNL AVX512_ICL AVX512_SPR"
# Do not set OPENBLAS_CORETYPE=Haswell: on Zen 4 (Ryzen 7950X3D) OpenBLAS 0.3.34 then segfaults
# in large float64 matmuls. Let OpenBLAS pick its kernel.
# JAX (GW solves) and PyTorch (φ/ψ training) share the GPU. The platform allocator returns JAX's
# memory after each solve, so torch does not run out afterwards. Must be set before JAX is imported.
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import sys
sys.path.append("..")

# Human–mouse alignment in 3D: a stack from one embryo, sections of the other placed into it

`human_mouse_align_merged.ipynb` aligns one HESTA section to one MOSTA section in 2D. The two sections are generally cut at different mediolateral depths. The laterality score only approximates the depth, so a 2D alignment has to explain the difference in depth as a deformation.

This notebook works in 3D instead:

1. **The stack.** The serial sections of one embryo are registered into a 3D stack. Adjacent sections of the same tissue allow strong constraints: each section moves rigidly, all sections share one embedding, and the gaps between them are estimated from the data (section 1).
2. **Placing sections.** Each section of the other dataset is placed into the stack (section 3):
   - *coarse*: one semi-relaxed fused GW from the 2D section to the whole 3D stack, and a plane fitted to where its mass lands;
   - *fine*: a search over cutting planes (offset z₀ and a small tilt) around it. The stack is cut along each plane into a virtual section, and the query is aligned to it in 2D.

   The final coupling is the alignment of the query with the best virtual section. Scores compare it with a 2D alignment to the laterality partner and to the best real stack section.

**Direction.** `DIRECTION = "human_stack"` builds the stack from the HESTA embryo and places the MOSTA sections; `"mouse_stack"` does the reverse. The demo runs both for CS18_E1 ↔ E12.5_E1, each as a papermill run of this notebook:

```
papermill human_mouse_align_3d.ipynb out_human_stack.ipynb -p DIRECTION human_stack
papermill human_mouse_align_3d.ipynb out_mouse_stack.ipynb -p DIRECTION mouse_stack
```

The last section compares the two runs once both have been run.

**Selection and evaluation stay separate.** Planes are chosen by the FGW objective only (spatial distances plus the ortholog cost). Organ labels and the held-out ortholog genes only score the result, as in the 2D notebook.

In [ ]:
# ============================================================
# Run settings (papermill parameters)
# ============================================================
DIRECTION = "human_stack"            # "human_stack": HESTA stack, MOSTA queries; "mouse_stack": the reverse
HUMAN_EMBRYO = ("HESTA", "CS18", "1")  # (dataset, stage, embryo) as in laterality_scores.csv
MOUSE_EMBRYO = ("MOSTA", "E12.5", "1")
STACK_SECTIONS = None                # None: every section of the stack embryo; or a list of section names
QUERY_SECTIONS = None                # None: every section of the query embryo; or a list of section names
EXCLUDE_SECTIONS = []                # sections to leave out (e.g. almost unlabelled ones)

ORDER  = None        # stack order: "file", "infer" (from the data), or None ("file" for HESTA, "infer" for MOSTA)
METHOD = "fgw"       # query alignments: "fgw" (fused GW on spatial distances, moscot's objective) or "mgw" (pullback
                     # geodesics; on the 3D stack in the coarse step)

# Points per section (merged bins, util.merge_bins)
MAX_OBS_STACK = 1000
MAX_OBS_QUERY = 1000
SUBSAMPLE, AGG, LLOYD_ITER = "voronoi", "sum", 3
DROP_LABELS = ["Cavity", "Extraembryonic Tissue"]

# Features
PCA_COMP     = 30
N_HVG        = 2000
N_EVAL_GENES = 500   # held-out genes that score the couplings (1:1 orthologs across species)

# Stage 1: the stack
PAIR_ALPHA    = 0.8      # spatial weight of the first FGW between neighbouring sections (moscot)
PAIR_EPS      = 1e-2
ICP_LAMBDA    = 0.5      # OT-ICP: weight of the registered spatial distance against the expression distance
ICP_ITER      = 15
ICP_EPS       = 5e-3
SECTION_FLIP  = "auto"   # "auto": per pair, rotation or reflection by the lower cost; False: rotations only
GLOBAL_SWEEPS = 2        # re-fit each section to both neighbours (limits drift)
Z_GAPS        = None     # None: estimated; or n_sections - 1 gaps in stack units (a section's RMS radius = 1)
LOO_STEPS     = 11       # t grid of the leave-one-out check

# Stage 2: placing the queries
FGW_ALPHA    = 0.5       # 2D query ↔ section alignments: spatial GW weight (1 - alpha on the ortholog cost)
FGW_EPS      = 5e-3
COARSE_ALPHA = 1.0       # direct 2D→3D FGW: weight of the ortholog cost (OTT fused_penalty)
COARSE_EPS   = 5e-3
TAU_STACK    = 0.1       # direct 2D→3D FGW: relaxation of the stack marginal (1 = balanced)
Z_WINDOW     = None      # half-width of the z₀ scan around the coarse z₀; None: the whole stack
N_Z          = 15
TILT_MAX     = 15        # degrees
N_TILT       = 3
MIN_COVERAGE = 0.6       # virtual sections covering less of a section than this are skipped
VIRTUAL_FEATURES = "rescaled"  # features of a virtual section: "rescaled" (mixed, length restored; unbiased),
                               # "cost" (interpolated cost; favours real sections), "linear" (favours in-between planes)
Z_SELECT     = "scan"    # final z0: "scan" (plane search over virtual sections) or "sections" (parabola through the
                         # objectives of the real sections; no tilt)
MIRROR_TEST  = True      # also try the mirror image of the query in every 2D alignment
SAMESPECIES_TEST = True  # place every interior stack section, held out, as a query; it should land at its own z

# MGW (METHOD = "mgw")
PHI_ARC      = (128, 256, 256, 128)
KNN_K        = 12
GEODESIC_EPS = 1e-1
LR           = 1e-3
N_ITER       = 20_000
FUSED_ALPHA  = 0.1
GW_PARAMS    = dict(verbose=False, inner_maxit=3000, outer_maxit=3000, inner_tol=1e-7, outer_tol=1e-7, epsilon=1e-4)
SEED         = 1

# Folders (None: <MGW data dir>/hesta, <MGW data dir>/mosta, <repo>/laterality, mgw/resources, <HESTA_DIR>/alignments/3d)
HESTA_DIR = MOSTA_DIR = LATERALITY_DIR = ORTHOLOG_TSV = SAVE_DIR = None

In [ ]:
import gc
import json
import logging
from datetime import datetime
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scanpy as sc
import torch
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr

import mgw as mgw_pkg
from mgw import crossspecies as cs, orthologs, stack3d as s3
from mgw import mgw
from mgw.config import get_data_dir
from mgw.gw import solve_gw_ott
from validation.moscot_align import align_moscot

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_dtype(torch.float64)
logging.getLogger("moscot._logging").setLevel(logging.WARNING)
print("Device:", device)

REPO = Path(mgw_pkg.__file__).resolve().parent.parent
DATA = get_data_dir()
HESTA_DIR = Path(HESTA_DIR or DATA / "hesta").expanduser()
if MOSTA_DIR is None:
    MOSTA_DIR = next((DATA / d for d in ["mosta", "mouse_embryo"] if (DATA / d).is_dir()), DATA / "mosta")
MOSTA_DIR = Path(MOSTA_DIR).expanduser()
LATERALITY_DIR = Path(LATERALITY_DIR or REPO / "laterality").expanduser()
ORTHO = orthologs.load_one2one(ORTHOLOG_TSV)
H2M = dict(zip(ORTHO.human, ORTHO.mouse))
SAVE_DIR = Path(SAVE_DIR or HESTA_DIR / "alignments" / "3d").expanduser()
SAVE_DIR.mkdir(parents=True, exist_ok=True)

scores = pd.read_csv(LATERALITY_DIR / "laterality_scores.csv", dtype={"embryo": str}).set_index("key")
STACK_EMB, QUERY_EMB = (HUMAN_EMBRYO, MOUSE_EMBRYO) if DIRECTION == "human_stack" else (MOUSE_EMBRYO, HUMAN_EMBRYO)
STACK_DS, QUERY_DS = STACK_EMB[0], QUERY_EMB[0]
emb_name = lambda e: f"{e[1]}_E{e[2]}"
stack_names = [s for s in (STACK_SECTIONS or cs.embryo_sections(scores, *STACK_EMB)) if s not in EXCLUDE_SECTIONS]
query_names = [s for s in (QUERY_SECTIONS or cs.embryo_sections(scores, *QUERY_EMB)) if s not in EXCLUDE_SECTIONS]
ORDER = ORDER or ("file" if STACK_DS == "HESTA" else "infer")
METHOD_NAME = {"mgw": "MGW", "fgw": "FGW"}[METHOD]
RUN_TAG = f"{emb_name(STACK_EMB)}_stack_{emb_name(QUERY_EMB)}_{METHOD}"
path = lambda ds, s: cs.section_path(ds, s, HESTA_DIR, MOSTA_DIR)

def lat(ds, s):
    k = cs.score_key(ds, s)
    return (float(scores.loc[k, "L_pair"]), float(scores.loc[k, "L_laterality"])) if k in scores.index else (np.nan, np.nan)

print(f"stack ({STACK_DS} {emb_name(STACK_EMB)}, order '{ORDER}'):", ", ".join(f"{s} (L {lat(STACK_DS, s)[0]:.2f})" for s in stack_names))
print(f"queries ({QUERY_DS} {emb_name(QUERY_EMB)}):", ", ".join(f"{s} (L {lat(QUERY_DS, s)[0]:.2f})" for s in query_names))
cs.check_files([(STACK_DS, s) for s in stack_names] + [(QUERY_DS, s) for s in query_names], HESTA_DIR, MOSTA_DIR)

## Helpers

**Coordinates.** Every section is centred and divided by a length scale. The stack uses one scale for all its sections: the median RMS radius of the sections. A query is scaled to an RMS radius of 1. The query and the stack sections then have comparable sizes, which removes the difference in absolute size between species. z is measured in the same units.

**Cross costs.** M is the distance between ortholog embeddings, scaled by one fixed number per query (the 0.99 quantile of its distances to the whole stack), so candidates of one query can be compared.

On a virtual section, a point at weight t between fᵢ and fⱼ needs features. How they are interpolated decides which planes look good (`VIRTUAL_FEATURES`):
- `"linear"`: (1 − t)fᵢ + t fⱼ. The mix of two different cells lies closer to the mean than either, and a query is closer to the mean on average, so planes *between* sections win.
- `"cost"`: the cost (1 − t)‖q − fᵢ‖ + t‖q − fⱼ‖. A query point that matches one endpoint well is pulled up by the other, so planes *through* real sections win.
- `"rescaled"` (default, `VirtualSection.interp_rescaled`): the linear mix, with its deviation from the stack mean rescaled to the mixed length (1 − t)‖fᵢ − m‖ + t‖fⱼ − m‖. Tiles are treated the same way. This removes the pull towards the mean without the pull towards real sections.

**2D alignments** (`align_2d`) take a cross cost M, not features:
- `METHOD = "fgw"`: fused GW, `FGW_ALPHA` × the GW term on spatial distances plus `1 − FGW_ALPHA` × ⟨M, P⟩. This is the objective of moscot's `AlignmentProblem`, solved with OTT directly because moscot takes features, not a cost matrix.
- `METHOD = "mgw"`: MGW, with geodesics from φ and ψ and M as its cross cost (`FUSED_ALPHA`).

`MIRROR_TEST` also aligns the mirror image of the query and keeps the better of the two.

**The objective** (`fgw_objective`) scores every 2D coupling the same way: `FGW_ALPHA` × the GW energy on the spatial distances (each side scaled to a maximum of 1) plus `1 − FGW_ALPHA` × ⟨M, P⟩. Planes are chosen by it. The stage-1 couplings between stack sections use moscot on the shared embedding.

In [ ]:
def centred(xy, scale=None):
    xy = np.asarray(xy, dtype=np.float64)
    xy = xy - xy.mean(0)
    return xy / (scale or s3.rms_radius(xy))


def light(xy, emb):
    """A minimal AnnData for the solvers: coordinates and one embedding."""
    a = ad.AnnData(obs=pd.DataFrame(index=[f"p{i}" for i in range(len(xy))]))
    a.obsm["spatial"], a.obsm["X_emb"], a.obsm["X_pca"] = np.asarray(xy, float), np.asarray(emb, float), np.asarray(emb, float)
    return a


def gw_energy(C1, C2, P):
    p, q = P.sum(1), P.sum(0)
    return float(p @ (C1 * C1) @ p + q @ (C2 * C2) @ q - 2 * ((C1 @ P) * (P @ C2)).sum())


def fgw_objective(xa, xb, M, P):
    Ca, Cb = cdist(xa, xa), cdist(xb, xb)
    P = P / P.sum()
    return FGW_ALPHA * gw_energy(Ca / Ca.max(), Cb / Cb.max(), P) + (1 - FGW_ALPHA) * float((M * P).sum())


def moscot_pair(xa, fa, xb, fb):
    """Stage 1: FGW of two stack sections on their shared embedding (moscot, spatial weight PAIR_ALPHA)."""
    P, _ = align_moscot(light(xa, fa), light(xb, fb), joint_attr="X_emb", alpha=PAIR_ALPHA, epsilon=PAIR_EPS,
                        verbose=False)
    return P


def solve_2d(xa, fa, xb, fb, M, method="fgw", tag=""):
    """Coupling (n_a x n_b) of two point sets with the cross cost M; fa, fb are the features MGW fits φ, ψ to."""
    if method == "fgw":
        Ca, Cb = cdist(xa, xa), cdist(xb, xb)
        return solve_gw_ott(Ca / Ca.max(), Cb / Cb.max(), M=M, alpha=(1 - FGW_ALPHA) / FGW_ALPHA, epsilon=FGW_EPS,
                            inner_maxit=3000, outer_maxit=200, verbose=False)
    pre = mgw.mgw_preprocess(light(xa, fa), light(xb, fb), PCA_comp=fa.shape[1], use_cca_feeler=False,
                             cross_key="X_emb", coords="isotropic", rep_norm="zscore", verbose=False)
    pre["M"] = np.asarray(M, dtype=np.float32)
    out = mgw.mgw_align_core(pre, widths=PHI_ARC, lr=LR, niter=N_ITER, knn_k=KNN_K, geodesic_eps=GEODESIC_EPS,
                             gw_params=GW_PARAMS, seed=SEED, deterministic=True, n_restarts=1,
                             fused_alpha=FUSED_ALPHA, verbose=False, tag=tag)
    P = np.asarray(out["P"])
    del out, pre
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return P


def align_2d(q, xb, fb, M, method="fgw", tag=""):
    """Align a query to a section; with MIRROR_TEST also its mirror image. Returns the better by the objective."""
    best = None
    for mirrored in ((False, True) if MIRROR_TEST else (False,)):
        xa = q["xy"] * [-1, 1] if mirrored else q["xy"]
        P = solve_2d(xa, q["emb"], xb, fb, M, method, tag=tag)
        obj = fgw_objective(xa, xb, M, P)
        if best is None or obj < best["obj"]:
            best = dict(P=P, obj=obj, mirrored=mirrored, det=cs.orientation_det(xa, xb, P) * (-1 if mirrored else 1))
    return best


def evaluate(P, q, labels_b, Y_b, full=True):
    """Label transfer and held-out gene co-expression (on the points, and on every bin of the query if full)."""
    lt = cs.label_transfer(P, q["labels"], labels_b)
    r, r0 = cs.coexpression(P, q["Y"], Y_b)
    out = dict(label_acc=lt["acc"], label_bal_acc=lt["bal_acc"], coexpr_r=float(np.nanmedian(r)),
               coexpr_r_shuffled=float(np.nanmedian(r0)), pred=lt["pred"], r=r, r0=r0)
    if full and q.get("Y_full") is not None:
        fs = cs.full_scores(P, q["full_obs"], labels_b, lt["pred"], Y_b, q["Y_full"])
        out.update(label_acc_full=fs["acc"], label_bal_acc_full=fs["bal_acc"], coexpr_r_full=fs["median_r"],
                   coexpr_r_shuffled_full=fs["median_r0"])
    return out


def plot_stack_3d(ST, palette, extra=(), title="", z_stretch=1.0):
    """The stack in 3D, one trace per organ class; `extra`: (xyz, colours, name) traces."""
    fig = go.Figure()
    coarse = np.where(np.isin(ST["labels"], list(palette)), ST["labels"], "Other")
    for c, col in palette.items():
        m = coarse == c
        if m.any():
            x = ST["xyz"][m]
            fig.add_trace(go.Scatter3d(x=x[:, 0], y=x[:, 1], z=x[:, 2] * z_stretch, mode="markers", name=c,
                                       marker=dict(size=2, color=col, opacity=0.35 if extra else 0.8)))
    for xyz, colours, name in extra:
        fig.add_trace(go.Scatter3d(x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2] * z_stretch, mode="markers", name=name,
                                   marker=dict(size=3.5, color=colours, line=dict(width=0.5, color="black"))))
    fig.update_layout(title=title, height=650, scene=dict(aspectmode="data", zaxis_title=f"z (×{z_stretch:g})"),
                      legend=dict(itemsizing="constant"))
    fig.show()

# 1. The stack

Each section is reduced to `MAX_OBS_STACK` points by merging bins (as in the 2D notebook). Only the reduced sections are kept, because the stack's full sections are not needed later. The features are one embedding over all stack sections: the most variable shared genes, z-scored within each section (which removes offsets between chips), then a joint PCA. The held-out genes of the cross-species evaluation are chosen in section 2 and excluded there; this embedding only registers the stack.

In [ ]:
stack_red = {}
for s in stack_names:
    red, full = cs.load_section(STACK_DS, s, path(STACK_DS, s), max_obs=MAX_OBS_STACK, subsample=SUBSAMPLE, agg=AGG,
                                lloyd_iter=LLOYD_ITER, drop_labels=DROP_LABELS)
    stack_red[s] = red
    del full
    gc.collect()

stack_ln = {s: cs.lognorm(a) for s, a in stack_red.items()}
for s, e in zip(stack_names, s3.joint_embedding([stack_ln[s] for s in stack_names], n_hvg=N_HVG, n_comps=PCA_COMP)):
    stack_red[s].obsm["X_feat"] = e
SCALE = float(np.median([s3.rms_radius(stack_red[s].obsm["spatial"]) for s in stack_names]))
xy_raw = {s: centred(stack_red[s].obsm["spatial"], SCALE) for s in stack_names}
print(f"stack length unit: {SCALE:.1f} pixels (median RMS radius of a section)")

## Section order

`ORDER = "file"` keeps the order of the section numbers. `"infer"` takes the order from the data. It compares every pair of sections by the objective of their FGW coupling (`PAIR_ALPHA`, at most 400 points per section). Then it finds the shortest path through all sections, which is exact (Held–Karp) for up to about 15 sections. MOSTA section numbers need not follow the physical order: E12.5_E1 has L = 0.76, 0.81, 0.10, 0.75, 0.67. The table prints the inferred order next to the file order and the laterality score.

In [ ]:
def sub(a, n=400, seed=0):
    idx = np.random.default_rng(seed).choice(a.n_obs, min(n, a.n_obs), replace=False)
    return idx

n_sec = len(stack_names)
D = np.zeros((n_sec, n_sec))
if ORDER == "infer" and n_sec > 2:
    for a in range(n_sec):
        for b in range(a + 1, n_sec):
            sa, sb = stack_names[a], stack_names[b]
            ia, ib = sub(stack_red[sa]), sub(stack_red[sb])
            xa, xb = xy_raw[sa][ia], xy_raw[sb][ib]
            fa, fb = stack_red[sa].obsm["X_feat"][ia], stack_red[sb].obsm["X_feat"][ib]
            P = moscot_pair(xa, fa, xb, fb)
            M = cdist(fa, fb); M /= np.quantile(M, 0.99)
            D[a, b] = D[b, a] = fgw_objective(xa, xb, M, P)
    order = s3.infer_order(D)
    if spearmanr(order, range(n_sec))[0] < 0:  # same direction as the file order where they agree
        order = order[::-1]
    print(f"path length: inferred {s3.path_length(D, order):.4f}, file order {s3.path_length(D, list(range(n_sec))):.4f}")
else:
    order = list(range(n_sec))
stack_names = [stack_names[i] for i in order]
print(pd.DataFrame({"stack position": range(n_sec), "section": stack_names,
                    "L_pair": [lat(STACK_DS, s)[0] for s in stack_names],
                    "L_raw": [lat(STACK_DS, s)[1] for s in stack_names]}).round(3).to_string(index=False))
if ORDER == "infer" and n_sec > 2:
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(D[np.ix_(order, order)], cmap="viridis")
    ax.set_xticks(range(n_sec), stack_names, rotation=60, ha="right"); ax.set_yticks(range(n_sec), stack_names)
    ax.set_title("FGW objective between sections (stack order)")
    fig.colorbar(im, ax=ax); plt.show()

## Rigid registration of neighbouring sections

Neighbours k and k+1 are first matched by FGW (moscot, spatial weight `PAIR_ALPHA`). OT-ICP then turns this into a rigid map (`stack3d.register_pair`). It alternates two steps:
- a weighted Procrustes fit on the coupling;
- a new linear OT on λ·‖x_k − (x_{k+1}R + t)‖² + (1 − λ)·‖f_k − f_{k+1}‖², with λ = `ICP_LAMBDA`.

Once the sections overlap, a point can only pair with nearby points of similar expression. `SECTION_FLIP = "auto"` runs ICP both as a rotation and as a reflection and keeps the lower cost, which catches a section mounted face down. The pairwise maps are composed into the frame of the middle section. `GLOBAL_SWEEPS` then re-fit every section to both its registered neighbours at once, which keeps errors from adding up along the stack.

In [ ]:
ref = n_sec // 2
pair_T, couplings, reg_rows = [], [], []
for k in range(n_sec - 1):
    sa, sb = stack_names[k], stack_names[k + 1]
    fa, fb = stack_red[sa].obsm["X_feat"], stack_red[sb].obsm["X_feat"]
    P0 = moscot_pair(xy_raw[sa], fa, xy_raw[sb], fb)
    det0 = cs.orientation_det(xy_raw[sb], xy_raw[sa], P0.T)
    reg = s3.register_pair(xy_raw[sa], xy_raw[sb], fa, fb, P0, flip=SECTION_FLIP, lam=ICP_LAMBDA,
                           n_iter=ICP_ITER, epsilon=ICP_EPS)
    pair_T.append(s3.homogeneous(reg["R"], reg["t"]))
    couplings.append(reg["P"])
    reg_rows.append(dict(pair=f"{sa} → {sb}", fgw_det=det0, reflected=reg["reflected"],
                         angle=np.degrees(np.arctan2(reg["R"][0, 1], reg["R"][0, 0])),
                         shift=float(np.linalg.norm(reg["t"])), rmsd=reg["rmsd"],
                         icp_cost_rotation=reg["cost_rotation"], icp_cost_reflection=reg["cost_reflection"],
                         icp_iters=len(reg["history"])))
T = s3.compose_to_reference(pair_T, ref)
flips = [bool(np.linalg.det(t[:2, :2]) < 0) for t in T]
if GLOBAL_SWEEPS:
    T = s3.global_sweeps([xy_raw[s] for s in stack_names], T, couplings, n_sweeps=GLOBAL_SWEEPS, ref=ref, flips=flips)
reg_xy = [s3.apply(T[k], xy_raw[s]) for k, s in enumerate(stack_names)]
print(pd.DataFrame(reg_rows).round(4).to_string(index=False))
if any(flips):
    print("mirrored relative to the reference:", [s for s, f in zip(stack_names, flips) if f])

ncol = min(n_sec - 1, 3)
fig, axes = plt.subplots(int(np.ceil((n_sec - 1) / ncol)), ncol, figsize=(5 * ncol, 4.2 * int(np.ceil((n_sec - 1) / ncol))),
                         squeeze=False)
for k, ax in enumerate(axes.ravel()):
    if k >= n_sec - 1:
        ax.axis("off"); continue
    ax.scatter(*reg_xy[k].T, s=3, c="C0", label=stack_names[k], linewidths=0)
    ax.scatter(*reg_xy[k + 1].T, s=3, c="C1", alpha=0.6, label=stack_names[k + 1], linewidths=0)
    ax.set_aspect("equal"); ax.axis("off"); ax.legend(markerscale=4, frameon=False, fontsize=8)
    ax.set_title(f"registered: rmsd {reg_rows[k]['rmsd']:.3f}")
plt.show()

## Gaps between the sections

The files hold no z. The gaps are estimated under local isotropy (`stack3d.estimate_gap`): expression should change with distance across sections as it does within a section.
1. The *in-plane variogram* γ(r) is the mean embedding distance between two points at in-plane distance r, pooled over the sections.
2. Each point of section k is paired with its nearest point of section k+1 in the registered frame, at in-plane offset rᵢ and embedding distance Dᵢ.
3. The gap d solves mean γ(√(rᵢ² + d²)) = mean Dᵢ.

A pair whose D lies above the variogram's range is *saturated*, and its gap is only a lower bound. Two checks are printed alongside, as shares of the total depth: the change in the raw laterality score |ΔL|, and (after the leave-one-out check below) the interpolation weight t* that best reproduces a held-out section. `Z_GAPS` overrides the estimate.

Caveats: anatomy changes faster along some axes than others, and two chips can differ beyond their mean offset. Either inflates D, so the estimate tends to be an upper bound.

In [ ]:
feats = [stack_red[s].obsm["X_feat"] for s in stack_names]
vg = s3.inplane_variogram(reg_xy, feats)
gap_rows = [dict(pair=f"{stack_names[k]} → {stack_names[k + 1]}",
                 **s3.estimate_gap(reg_xy[k], reg_xy[k + 1], feats[k], feats[k + 1], vg),
                 dL_raw=abs(lat(STACK_DS, stack_names[k + 1])[1] - lat(STACK_DS, stack_names[k])[1]))
            for k in range(n_sec - 1)]
gaps_df = pd.DataFrame(gap_rows)
gaps = list(Z_GAPS) if Z_GAPS is not None else gaps_df.gap.tolist()
if Z_GAPS is not None:
    gaps_df["gap_used"] = gaps
z_sec = np.r_[0.0, np.cumsum(gaps)]
gaps_df["share_gap"] = np.asarray(gaps) / max(sum(gaps), 1e-12)
gaps_df["share_dL"] = gaps_df.dL_raw / max(gaps_df.dL_raw.sum(), 1e-12)
print(gaps_df.round(4).to_string(index=False))
print("z of the sections:", dict(zip(stack_names, z_sec.round(3))))

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(vg.r, vg.gamma, "k-", label="in-plane variogram γ(r)")
for k, row in gaps_df.iterrows():
    ax.axhline(row.D, color=f"C{k}", ls="--", lw=1, label=f"D: {row.pair}")
ax.set_xlabel("in-plane distance (stack units)"); ax.set_ylabel("mean embedding distance")
ax.legend(frameon=False, fontsize=7); plt.show()

## The stack in 3D

The stack holds every point of every section, with (x, y) in the frame of the reference section and z from the gaps. Each coupling of neighbouring sections is kept as sparse *displacement pairs*: the peak partner of every row and of every column, weighted by its mass. These pairs connect the sections. Virtual sections are cut along them, and MGW's 3D kNN graph uses them as edges between sections.

In [ ]:
offsets = np.r_[0, np.cumsum([stack_red[s].n_obs for s in stack_names])]
ST = dict(
    names=stack_names, z=z_sec,
    section=np.concatenate([[s] * stack_red[s].n_obs for s in stack_names]),
    xyz=np.vstack([np.c_[reg_xy[k], np.full(len(reg_xy[k]), z_sec[k])] for k in range(n_sec)]),
    labels=np.concatenate([stack_red[s].obs["label_h"].to_numpy(str) for s in stack_names]),
    emb={"X_feat": np.vstack(feats)},
    pairs=[s3.displacement_pairs(couplings[k], offsets[k], offsets[k + 1]) for k in range(n_sec - 1)],
)
c, u, v = s3.principal_axes(ST["xyz"][:, :2])
ST["frame"] = dict(centre=c, u=u, v=v)
PALETTE = cs.class_palette(ST["labels"])
# stretch z for display when the stack is thin compared with a section (at most 4x)
z_stretch = float(np.clip(0.5 * np.ptp(ST["xyz"][:, :2], axis=0).max() / max(np.ptp(z_sec), 1e-12), 1, 4))
plot_stack_3d(ST, PALETTE, title=f"{emb_name(STACK_EMB)}: {n_sec} sections", z_stretch=z_stretch)

## Leave-one-out check of the stack

For every interior section k, the neighbours k−1 and k+1 are registered to each other directly (FGW, then OT-ICP). Virtual sections are cut between them at a grid of interpolation weights t, and each one is compared with the real section k: each tile takes the label and expression of its nearest point of section k. The scores are label agreement and the median r over the 200 most variable genes of the stack. The endpoints t = 0 and t = 1 are the baselines that copy a neighbour.

If interpolation works, an interior t beats both endpoints. The best t* also estimates the position of k between its neighbours, which can be compared with the gap estimate. The same coupling serves the same-species placement test in section 3.

In [ ]:
stack_hvg = pd.Index(stack_ln[stack_names[0]].var_names)
for s in stack_names[1:]:
    stack_hvg = stack_hvg.intersection(stack_ln[s].var_names)
Xcat = ad.concat([stack_ln[s][:, stack_hvg] for s in stack_names])
hv = stack_hvg[np.argsort(-cs.gene_var(Xcat.X))[:200]]
Y_stack_hvg = cs.dense(Xcat[:, hv].X)
del Xcat

loo_rows, loo_pairs = [], {}
for k in range(1, n_sec - 1):
    a, b = k - 1, k + 1
    P0 = moscot_pair(reg_xy[a], feats[a], reg_xy[b], feats[b])
    reg = s3.register_pair(reg_xy[a], reg_xy[b], feats[a], feats[b], P0, flip=False, lam=ICP_LAMBDA,
                           n_iter=ICP_ITER, epsilon=ICP_EPS)
    loo_pairs[k] = reg["P"]
    # both neighbours already sit in the stack frame; the direct coupling only supplies the pairs
    xyz2 = np.vstack([np.c_[reg_xy[a], np.zeros(len(reg_xy[a]))], np.c_[reg_xy[b], np.ones(len(reg_xy[b]))]])
    pairs2 = [s3.displacement_pairs(reg["P"], 0, len(reg_xy[a]))]
    Y2 = np.vstack([Y_stack_hvg[offsets[a]:offsets[a + 1]], Y_stack_hvg[offsets[b]:offsets[b + 1]]])
    lab2 = np.r_[ST["labels"][offsets[a]:offsets[a + 1]], ST["labels"][offsets[b]:offsets[b + 1]]]
    tree = cKDTree(reg_xy[k])
    Yk, labk = Y_stack_hvg[offsets[k]:offsets[k + 1]], ST["labels"][offsets[k]:offsets[k + 1]]
    for t in np.linspace(0, 1, LOO_STEPS):
        vs = s3.VirtualSection(xyz2, pairs2, s3.plane(t), ST["frame"], max_obs=MAX_OBS_STACK)
        nn = tree.query(vs.xyz[:, :2])[1]
        loo_rows.append(dict(section=stack_names[k], t=t, label_agree=float((vs.labels(lab2) == labk[nn]).mean()),
                             median_r=float(np.nanmedian(cs.colwise_r(vs.interp(Y2), Yk[nn]))),
                             t_gaps=gaps[a] / max(gaps[a] + gaps[k], 1e-12)))
loo = pd.DataFrame(loo_rows)
if len(loo):
    best_t = loo.loc[loo.groupby("section").median_r.idxmax(), ["section", "t", "t_gaps"]].rename(columns={"t": "t_star"})
    ends = loo[loo.t.isin([0, 1])].groupby("section")[["label_agree", "median_r"]].max().add_suffix("_best_end")
    mid = loo.groupby("section")[["label_agree", "median_r"]].max().add_suffix("_best_t")
    print(best_t.set_index("section").join(mid).join(ends).round(3).to_string())
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
    for s, g in loo.groupby("section", sort=False):
        ax[0].plot(g.t, g.label_agree, marker="o", ms=3, label=s); ax[1].plot(g.t, g.median_r, marker="o", ms=3)
        ax[1].axvline(g.t_gaps.iloc[0], color=ax[1].lines[-1].get_color(), ls=":", lw=1)
    ax[0].set_ylabel("label agreement"); ax[1].set_ylabel("median gene r (200 HVGs)")
    for a_ in ax: a_.set_xlabel("t (0: section k−1, 1: section k+1)")
    ax[0].legend(frameon=False, fontsize=7); ax[1].set_title("dotted: t from the gap estimate", fontsize=9)
    plt.show()
else:
    print("fewer than three sections: no leave-one-out check")

In [ ]:
stack_ad = ad.AnnData(obs=pd.DataFrame({"section": ST["section"], "label_h": ST["labels"]},
                                       index=np.concatenate([stack_red[s].obs_names for s in stack_names])))
stack_ad.obsm["spatial3d"], stack_ad.obsm["X_feat"] = ST["xyz"], ST["emb"]["X_feat"]
stack_ad.uns.update(names=stack_names, z=z_sec, scale=SCALE, order_mode=ORDER, ref=ref,
                    centre=c, u=u, v=v, transforms=np.stack(T))
stack_ad.write_h5ad(SAVE_DIR / f"stack_{emb_name(STACK_EMB)}_{SUBSAMPLE}{MAX_OBS_STACK}.h5ad")
np.savez(SAVE_DIR / f"stack_{emb_name(STACK_EMB)}_{SUBSAMPLE}{MAX_OBS_STACK}_pairs.npz",
         **{f"pairs_{k}": np.c_[i, j, w] for k, (i, j, w) in enumerate(ST["pairs"])})
with open(SAVE_DIR / f"order_{emb_name(STACK_EMB)}.json", "w") as fh:
    json.dump(dict(order=stack_names, mode=ORDER, z=z_sec.tolist(), date=datetime.now().isoformat(timespec="seconds")), fh)
print("saved the stack to", SAVE_DIR)

# 2. Query sections and cross-species features

The query sections are reduced to `MAX_OBS_QUERY` points. Their full sections are reloaded later, one at a time, for the full-resolution scores.

The features shared with the stack are the ortholog embedding of the 2D notebook (`orthologs.shared_embedding`), over all stack points and all query points at once. Genes are z-scored within each section, so offsets between chips and between species drop out.

The held-out evaluation genes are the `N_EVAL_GENES` most variable 1:1 orthologs (mitochondrial genes excluded). They are kept out of the embedding.

In [ ]:
query_red, query_full_obs = {}, {}
for s in query_names:
    red, full = cs.load_section(QUERY_DS, s, path(QUERY_DS, s), max_obs=MAX_OBS_QUERY, subsample=SUBSAMPLE, agg=AGG,
                                lloyd_iter=LLOYD_ITER, drop_labels=DROP_LABELS)
    query_red[s] = red
    query_full_obs[s] = ad.AnnData(obs=full.obs[["label_h", "tile"]].copy())
    del full
    gc.collect()
query_ln = {s: cs.lognorm(a) for s, a in query_red.items()}

stack_cat = ad.concat([stack_ln[s] for s in stack_names], join="inner")
query_cat = ad.concat([query_ln[s] for s in query_names], join="inner")
stack_groups, query_groups = ST["section"], np.concatenate([[s] * query_red[s].n_obs for s in query_names])
human, mouse, g_h, g_m = ((stack_cat, query_cat, stack_groups, query_groups) if STACK_DS == "HESTA"
                          else (query_cat, stack_cat, query_groups, stack_groups))
gh, gm = orthologs.restrict_to_orthologs(human.var_names, mouse.var_names, ORTHO)
eval_h = cs.pick_eval_genes(human[:, gh].X, mouse[:, gm].X, gh, N_EVAL_GENES)
eval_m = np.array([H2M[g] for g in eval_h])
E_h, E_m, emb_genes = orthologs.shared_embedding(human, mouse, ORTHO, n_hvg=N_HVG, n_comps=PCA_COMP, exclude=eval_h,
                                                 groups_A=g_h, groups_B=g_m)
E_stack, E_query = (E_h, E_m) if STACK_DS == "HESTA" else (E_m, E_h)
eval_stack, eval_query = (eval_h, eval_m) if STACK_DS == "HESTA" else (eval_m, eval_h)
ST["emb"]["X_ortho"] = E_stack
ST["Y"] = cs.dense(stack_cat[:, eval_stack].X)
q_off = np.r_[0, np.cumsum([query_red[s].n_obs for s in query_names])]
QUERIES = {}
for n, s in enumerate(query_names):
    QUERIES[s] = dict(name=s, xy=centred(query_red[s].obsm["spatial"]), emb=E_query[q_off[n]:q_off[n + 1]],
                      labels=query_red[s].obs["label_h"].to_numpy(str), Y=cs.dense(query_ln[s][:, eval_query].X),
                      full_obs=query_full_obs[s], L=lat(QUERY_DS, s)[0])
print(f"{len(gh):,} 1:1 orthologs in both; embedding on {len(emb_genes):,} genes; {len(eval_h)} evaluation genes")
del stack_cat, query_cat, human, mouse
_ = gc.collect()

# 3. Placing the queries

**Coarse: direct 2D→3D FGW.**
- Source: the query with its 2D distances. Target: every stack point with its 3D distances. Plus `COARSE_ALPHA` times the ortholog cost.
  - `METHOD = "fgw"`: Euclidean distances, both sides in stack units.
  - `METHOD = "mgw"`: φ is fit on the query in 2D and ψ on the stack in 3D, and the geodesics run on a kNN graph within each section plus the displacement pairs between sections.
- The query marginal is fixed. The stack marginal is relaxed (`TAU_STACK`), so the query can take up one slab of the stack instead of being spread over every section.
- A weighted PCA of the query's barycentric image in 3D gives a plane: z₀ at the stack centre, two tilts, and the RMS distance of the image from the plane.

**Fine: plane search.** Virtual sections are cut along candidate planes (`stack3d.VirtualSection`): every displacement pair is a segment between two neighbouring sections, and where it crosses the plane it gives a point. The query is aligned to each virtual section in 2D (FGW with the interpolated cross cost, plus its mirror image if `MIRROR_TEST`) and scored by the objective.
1. Scan z₀ (`N_Z` steps over the stack, or ±`Z_WINDOW` around the coarse z₀) at zero tilt.
2. Scan a tilt grid (`N_TILT`² angles up to ±`TILT_MAX`°) at the best z₀.
3. Re-scan z₀ at half the step around the best plane.

Planes covering less than `MIN_COVERAGE` of a section (they leave the stack) are skipped. A best plane at the end of the stack is flagged `edge`: the query may lie outside the range the stack covers.

The final coupling aligns the query with the virtual section of the best plane, using `METHOD`. It is scored by label transfer and held-out gene co-expression, on the points and on every bin of the query. Two 2D alignments serve as comparison: with the stack section closest in laterality score, and with the real stack section of lowest objective.

**Side.** The stack's orientation is fixed, so a query from the other side of the midline matches it as a mirror image: `det = -1` means the query is mirrored relative to the stack.

In [ ]:
def coarse_place(q, ST, emb_key):
    M = cdist(q["emb"], ST["emb"][emb_key]); M /= np.quantile(M, 0.99)
    if METHOD == "mgw":
        S = light(ST["xyz"], ST["emb"][emb_key]); S.obsm["spatial3d"] = ST["xyz"]
        pre = mgw.mgw_preprocess(light(q["xy"], q["emb"]), S, PCA_comp=q["emb"].shape[1], use_cca_feeler=False,
                                 cross_key="X_emb", coords="isotropic", rep_norm="zscore", spatial_key_B="spatial3d",
                                 verbose=False)
        pre["graph_N"] = s3.stack_graph(pre["xs2"], ST["section"], ST["pairs"], k_in=KNN_K)
        out = mgw.mgw_align_core(pre, widths=PHI_ARC, lr=LR, niter=N_ITER, knn_k=KNN_K, geodesic_eps=GEODESIC_EPS,
                                 gw_params={**GW_PARAMS, "epsilon": COARSE_EPS, "tau_b": TAU_STACK}, seed=SEED, deterministic=True,
                                 n_restarts=1, fused_alpha=COARSE_ALPHA, verbose=False)
        P = np.asarray(out["P"])
        del out, pre
    else:
        Cs = cdist(ST["xyz"], ST["xyz"]); s = Cs.max()
        P = solve_gw_ott(cdist(q["xy"], q["xy"]) / s, Cs / s, M=M, alpha=COARSE_ALPHA, epsilon=COARSE_EPS,
                         tau_a=1.0, tau_b=TAU_STACK, inner_maxit=3000, outer_maxit=200, verbose=False)
        del Cs
    gc.collect()
    fit = s3.plane_from_coupling(ST["xyz"], P, ST["frame"], section=ST["section"])
    xy_img, _ = s3.plane_coords(fit["bary"], fit["plane"], ST["frame"])
    fit["det"] = cs.orientation_det(q["xy"], xy_img, np.eye(len(xy_img)))
    fit["P"] = P
    return fit


def virtual(q, ST, pl, emb_key, M_scale):
    """Virtual section of a plane, its interpolated features and the query's cross cost (None if too small)."""
    vs = s3.VirtualSection(ST["xyz"], ST["pairs"], pl, ST["frame"], max_obs=MAX_OBS_QUERY)
    if vs.n_obs < 50 or vs.coverage < MIN_COVERAGE:
        return vs, None, None
    F = ST["emb"][emb_key]
    if VIRTUAL_FEATURES == "cost":
        return vs, vs.interp(F), vs.interp_cost(q["emb"], F) / M_scale
    fv = vs.interp(F) if VIRTUAL_FEATURES == "linear" else vs.interp_rescaled(F)
    return vs, fv, cdist(q["emb"], fv) / M_scale


def scan(q, ST, emb_key, planes, M_scale, mirror=None):
    """Align q to the virtual section of each plane (FGW); one row per plane."""
    rows = []
    for pl in planes:
        vs, fv, M = virtual(q, ST, pl, emb_key, M_scale)
        row = dict(**pl, coverage=vs.coverage, n_virtual=vs.n_obs)
        if fv is not None:
            if mirror is None:
                res = align_2d(q, vs.xy, fv, M)
            else:
                xa = q["xy"] * [-1, 1] if mirror else q["xy"]
                P = solve_2d(xa, q["emb"], vs.xy, fv, M)
                res = dict(P=P, obj=fgw_objective(xa, vs.xy, M, P), mirrored=mirror,
                           det=cs.orientation_det(xa, vs.xy, P) * (-1 if mirror else 1))
            ev = evaluate(res["P"], q, vs.labels(ST["labels"]), vs.interp(ST["Y"]), full=False)
            row.update(objective=res["obj"], mirrored=res["mirrored"], det=res["det"],
                       label_acc=ev["label_acc"], coexpr_r=ev["coexpr_r"])
        rows.append(row)
    return pd.DataFrame(rows)


def place_query(q, ST, emb_key="X_ortho", final_method=METHOD, compare=True, plots=True):
    """Coarse plane, plane search, final alignment and scores for one query. Returns (summary row, artifacts)."""
    name = q["name"]
    M_scale = np.quantile(cdist(q["emb"], ST["emb"][emb_key]), 0.99)
    z_lo, z_hi = ST["z"].min(), ST["z"].max()

    # coarse
    co = coarse_place(q, ST, emb_key)
    z_c = co["plane"]["z0"]
    print(f"[{name}] coarse plane z0 {z_c:.3f} (stack {z_lo:.3f}–{z_hi:.3f}), tilt "
          f"{co['plane']['theta_u']:+.1f}°/{co['plane']['theta_v']:+.1f}°, planarity rms {co['rms']:.3f}, det {co['det']:+d}")

    # real sections: one 2D alignment each, and z from the parabola through the best one and its neighbours
    real_obj = {}
    for s in ST["names"]:
        m = ST["section"] == s
        real_obj[s] = align_2d(q, ST["xyz"][m, :2], ST["emb"][emb_key][m], cdist(q["emb"], ST["emb"][emb_key][m]) / M_scale)
    z_s, edge_s = s3.z_from_sections(ST["z"], [real_obj[s]["obj"] for s in ST["names"]])

    # fine: z scan, tilt grid, z refine
    lo, hi = (z_lo, z_hi) if Z_WINDOW is None else (max(z_lo, z_c - Z_WINDOW), min(z_hi, z_c + Z_WINDOW))
    zs = np.linspace(lo, hi, N_Z) if hi > lo else np.array([lo])
    zs = np.unique(np.r_[zs, ST["z"][(ST["z"] >= lo) & (ST["z"] <= hi)]])  # the real sections' z as well
    sz = scan(q, ST, emb_key, [s3.plane(z) for z in zs], M_scale).assign(stage="z scan")
    if sz.objective.notna().sum() == 0:
        print(f"[{name}] no virtual section with coverage >= {MIN_COVERAGE}")
        return None, None
    b = sz.loc[sz.objective.idxmin()]
    mirror = bool(b.mirrored) if MIRROR_TEST else False
    tilts = np.linspace(-TILT_MAX, TILT_MAX, N_TILT) if N_TILT > 1 else np.array([0.0])
    st = scan(q, ST, emb_key, [s3.plane(b.z0, tu, tv) for tu in tilts for tv in tilts if (tu, tv) != (0, 0)],
              M_scale, mirror).assign(stage="tilt grid")
    allr = pd.concat([sz, st], ignore_index=True)
    b = allr.loc[allr.objective.idxmin()]
    dz = (hi - lo) / max(N_Z - 1, 1) / 2
    if dz > 0:
        sr = scan(q, ST, emb_key, [s3.plane(z, b.theta_u, b.theta_v) for z in (b.z0 - dz, b.z0 + dz) if lo <= z <= hi],
                  M_scale, mirror).assign(stage="z refine")
        allr = pd.concat([allr, sr], ignore_index=True)
    b_scan = allr.loc[allr.objective.idxmin()]
    edge_scan = bool(np.isclose(b_scan.z0, z_lo) or np.isclose(b_scan.z0, z_hi))
    b, edge = b_scan, edge_scan
    if Z_SELECT == "sections":
        bs = scan(q, ST, emb_key, [s3.plane(z_s)], M_scale).assign(stage="sections").iloc[0]
        if pd.notna(bs.get("objective")):
            b, edge = bs, edge_s
        else:
            print(f"[{name}] the virtual section at z0 {z_s:.3f} is too small; using the scan")
    best_pl = s3.plane(b.z0, b.theta_u, b.theta_v)

    # final alignment at the best plane
    vs, fv, M = virtual(q, ST, best_pl, emb_key, M_scale)
    lab_v, Y_v = vs.labels(ST["labels"]), vs.interp(ST["Y"])
    xa = q["xy"] * [-1, 1] if b.mirrored else q["xy"]
    P = solve_2d(xa, q["emb"], vs.xy, fv, M, final_method, tag=f"{name}_3d")
    ev = evaluate(P, q, lab_v, Y_v)
    row = dict(query=name, L_query=q["L"], z0=b.z0, theta_u=b.theta_u, theta_v=b.theta_v, coverage=b.coverage,
               edge=edge, mirrored=bool(b.mirrored), det=int(b.det), objective=fgw_objective(xa, vs.xy, M, P),
               z0_coarse=z_c, theta_u_coarse=co["plane"]["theta_u"], theta_v_coarse=co["plane"]["theta_v"],
               planarity_rms=co["rms"], det_coarse=co["det"],
               z0_scan=b_scan.z0, theta_u_scan=b_scan.theta_u, theta_v_scan=b_scan.theta_v, edge_scan=edge_scan,
               z0_sections=z_s, edge_sections=edge_s,
               nearest_section=ST["names"][int(np.argmin(np.abs(ST["z"] - b.z0)))],
               **{k: v for k, v in ev.items() if k not in ("pred", "r", "r0")})

    # comparisons: the laterality partner and the best real section (2D, FGW to choose, then METHOD)
    comp = {}
    if compare:
        L_stack = {s: lat(STACK_DS, s)[0] for s in ST["names"]}
        L_ok = {s: l for s, l in L_stack.items() if np.isfinite(l)}
        L_partner = min(L_ok, key=lambda s: abs(L_ok[s] - q["L"])) if np.isfinite(q["L"]) and L_ok else None
        best_real = min(real_obj, key=lambda s: real_obj[s]["obj"])
        for kind, s in (("L partner", L_partner), ("best real section", best_real)):
            if s is None:
                continue
            m = ST["section"] == s
            P2 = real_obj[s]["P"]
            if final_method != "fgw":
                xa2 = q["xy"] * [-1, 1] if real_obj[s]["mirrored"] else q["xy"]
                Mr = cdist(q["emb"], ST["emb"][emb_key][m]) / M_scale
                P2 = solve_2d(xa2, q["emb"], ST["xyz"][m, :2], ST["emb"][emb_key][m], Mr, final_method, tag=f"{name}_{s}")
            ev2 = evaluate(P2, q, ST["labels"][m], ST["Y"][m])
            comp[kind] = dict(section=s, objective=real_obj[s]["obj"], P=P2,
                              **{k: v for k, v in ev2.items() if k not in ("pred", "r", "r0")})
            row.update({f"{kind.replace(' ', '_')}_{k}": v for k, v in comp[kind].items() if k != "P"})
    row["real_section_objectives"] = json.dumps({s: round(r["obj"], 5) for s, r in real_obj.items()})

    if plots:
        fig, ax = plt.subplots(1, 3, figsize=(18, 3.6), gridspec_kw=dict(width_ratios=[2.2, 1, 1.3]))
        g = allr[allr.stage != "tilt grid"].sort_values("z0")
        g0 = g[(g.theta_u == 0) & (g.theta_v == 0)]
        ax[0].plot(g0.z0, g0.objective, "k-o", ms=3, label="objective (virtual, tilt 0)")
        ax[0].plot(ST["z"], [real_obj[s]["obj"] for s in ST["names"]], "s", color="C3", ms=6, label="real sections")
        ax[0].axvline(b_scan.z0, color="C2", lw=1.5, label=f"scan z0 {b_scan.z0:.3f}")
        ax[0].axvline(z_s, color="C3", ls=":", lw=1.5, label=f"sections z0 {z_s:.3f} (parabola)")
        ax[0].axvline(z_c, color="C1", ls="--", lw=1, label=f"coarse z0 {z_c:.3f}")
        ax[0].set_xlabel("z0"); ax[0].set_ylabel("FGW objective")
        a2 = ax[0].twinx()
        a2.plot(g0.z0, g0.label_acc, ":", color="C0", label="label acc (eval)")
        a2.plot(g0.z0, g0.coexpr_r, ":", color="C4", label="median r (eval)")
        a2.set_ylabel("evaluation")
        h1, l1 = ax[0].get_legend_handles_labels(); h2, l2 = a2.get_legend_handles_labels()
        ax[0].legend(h1 + h2, l1 + l2, frameon=False, fontsize=7, loc="upper left", bbox_to_anchor=(1.12, 1))
        ax[0].set_title(f"{name}: plane search")
        tg = allr[allr.stage == "tilt grid"]
        if len(tg):
            H = pd.concat([tg, g0[np.isclose(g0.z0, tg.z0.iloc[0])]]).pivot_table(index="theta_v", columns="theta_u",
                                                                                  values="objective")
            im = ax[1].imshow(H.to_numpy(), cmap="viridis_r", origin="lower")
            ax[1].set_xticks(range(H.shape[1]), [f"{x:g}" for x in H.columns]); ax[1].set_yticks(range(H.shape[0]), [f"{x:g}" for x in H.index])
            ax[1].set_xlabel("θ_u (°)"); ax[1].set_ylabel("θ_v (°)"); ax[1].set_title(f"tilt at z0 {tg.z0.iloc[0]:.3f}")
            fig.colorbar(im, ax=ax[1])
        else:
            ax[1].axis("off")
        if co["mass_per_section"] is not None:
            ms = co["mass_per_section"].reindex(ST["names"])
            ax[2].bar(range(len(ms)), ms.to_numpy(), color="C1")
            ax[2].set_xticks(range(len(ms)), [f"{s}\nz {z:.2f}" for s, z in zip(ST["names"], ST["z"])], rotation=60, ha="right", fontsize=7)
            ax[2].set_ylabel("share of transported mass"); ax[2].set_title("coarse 2D→3D coupling")
        plt.tight_layout(); plt.show()

        cs.plot_label_panels([(vs.xy, lab_v, f"virtual section z0 {b.z0:.3f}"), (q["xy"], q["labels"], f"{name}: annotated"),
                              (q["xy"], ev["pred"], f"{name}: transferred from the stack")], PALETTE)
        cs.plot_coupling_lines(xa, vs.xy, P, f"{METHOD_NAME}: {name} → virtual section", (name, "virtual"))
        q3 = (P @ vs.xyz) / (P.sum(1, keepdims=True) + 1e-30)
        plot_stack_3d(ST, PALETTE, extra=[(co["bary"], cs.colors_for(q["labels"], PALETTE), f"{name}: coarse image"),
                                          (q3, cs.colors_for(q["labels"], PALETTE), f"{name}: final placement")],
                      title=f"{name} in {emb_name(STACK_EMB)}", z_stretch=z_stretch)
        print(pd.DataFrame([{k: row[k] for k in ("objective", "label_acc", "label_bal_acc", "coexpr_r", "coexpr_r_shuffled",
                                                  "label_acc_full", "coexpr_r_full") if k in row}]
                           + [{k: v for k, v in comp[c].items() if k not in ("section", "P")} for c in comp],
                           index=["3D placement"] + [f"{c} ({comp[c]['section']})" for c in comp]).round(3).to_string())
    return row, dict(P=P, plane=best_pl, scan=allr, coarse=co, virtual=vs,
                     comparisons={k.replace(" ", "_"): (c["section"], c["P"]) for k, c in comp.items()})

## Same-species check: held-out stack sections placed as queries

Each interior section in turn is removed from the stack. Its neighbours are joined by their direct coupling from the leave-one-out check, and the section is placed back with the stack's own features (`X_feat`). It should land at its own z, with tilt about 0, `det = +1` and full coverage. The offsets from the true z measure how well the pipeline places a section in the best case, for both z estimates:
- **scan**: the plane search over virtual sections;
- **sections**: the parabola through the objectives of the real sections.

The "true" z comes from the gap estimates, which are themselves uncertain (saturated gaps are lower bounds). The table therefore also gives the position as a share of the gap between the neighbours (t: 0 at the lower neighbour, 1 at the upper), next to t* of the leave-one-out check. Plots are shown for the middle section only.

In [ ]:
def drop_section(ST, k, P_loo):
    keep = ST["section"] != ST["names"][k]
    new_idx = np.full(len(keep), -1); new_idx[keep] = np.arange(keep.sum())
    pairs = []
    for kk, (i, j, w) in enumerate(ST["pairs"]):
        if kk == k - 1:   # (k-1, k) and (k, k+1) become one pair (k-1, k+1)
            off_a, off_b = np.flatnonzero(ST["section"] == ST["names"][k - 1])[0], np.flatnonzero(ST["section"] == ST["names"][k + 1])[0]
            i2, j2, w2 = s3.displacement_pairs(P_loo, off_a, off_b)
            pairs.append((new_idx[i2], new_idx[j2], w2))
        elif kk != k:
            pairs.append((new_idx[i], new_idx[j], w))
    out = dict(ST, names=[s for s in ST["names"] if s != ST["names"][k]], z=np.delete(ST["z"], k), section=ST["section"][keep],
               xyz=ST["xyz"][keep], labels=ST["labels"][keep], emb={e: x[keep] for e, x in ST["emb"].items()},
               Y=Y_stack_hvg[keep], pairs=pairs)
    return out

ss_rows = []
if SAMESPECIES_TEST and loo_pairs:
    k_plot = sorted(loo_pairs, key=lambda kk: abs(kk - (n_sec - 1) / 2))[0]
    t_star = best_t.set_index("section").t_star if len(loo) else pd.Series(dtype=float)
    for k in sorted(loo_pairs):
        s = stack_names[k]
        m = ST["section"] == s
        q_ss = dict(name=f"{s} (held out)", xy=centred(stack_red[s].obsm["spatial"]), emb=ST["emb"]["X_feat"][m],
                    labels=ST["labels"][m], Y=Y_stack_hvg[m], full_obs=None, L=lat(STACK_DS, s)[0])
        ST_minus = drop_section(dict(ST, Y=None), k, loo_pairs[k])
        # the query is scaled to an RMS radius of 1, the stack by its median; undo the difference for this check
        q_ss["xy"] *= s3.rms_radius(xy_raw[s])
        r, _ = place_query(q_ss, ST_minus, emb_key="X_feat", final_method="fgw", compare=False, plots=(k == k_plot))
        if r is None:
            continue
        za, zb = z_sec[k - 1], z_sec[k + 1]
        tt = lambda zz: (zz - za) / (zb - za)
        ss_rows.append(dict(section=s, true_z=z_sec[k], z0_scan=r["z0_scan"], z0_sections=r["z0_sections"],
                            z0_coarse=r["z0_coarse"], t_true=tt(z_sec[k]), t_scan=tt(r["z0_scan"]),
                            t_sections=tt(r["z0_sections"]), t_star_loo=t_star.get(s, np.nan),
                            tilt=f"{r['theta_u_scan']:+.0f}/{r['theta_v_scan']:+.0f}", det=r["det"],
                            label_acc=r["label_acc"]))
    ss = pd.DataFrame(ss_rows)
    if len(ss):
        print(ss.round(3).to_string(index=False))
        for e in ("scan", "sections", "coarse"):
            print(f"mean |z0 - true z| ({e}): {np.mean(np.abs(ss[f'z0_{e}'] - ss.true_z)):.3f}")
        print(f"mean |z of the nearest neighbour - true z|: "
              f"{np.mean([min(z_sec[k] - z_sec[k - 1], z_sec[k + 1] - z_sec[k]) for k in sorted(loo_pairs)]):.3f}")

## The queries

One block of plots per query:
1. **Plane search**:
   - the objective along z₀ (black), with the real stack sections as red squares and the evaluation scores as dotted lines;
   - the tilt grid at the best z₀;
   - the share of the coarse coupling's mass that lands on each section.
2. **Labels** on the virtual section, the query, and the query after transfer from the stack.
3. **The coupling** between the query and the virtual section.
4. **3D view** of the query in the stack: its coarse barycentric image and its final placement.
5. **Scores** of the 3D placement against the 2D alignments with the laterality partner and with the best real section.

In [ ]:
rows, results = [], {}
for s in query_names:
    row, res = place_query(QUERIES[s], ST)
    if row is not None:
        rows.append(row)
        results[s] = res
        np.savez(SAVE_DIR / f"P_{s}_in_{emb_name(STACK_EMB)}_{METHOD}.npz", P=res["P"].astype(np.float32),
                 plane=np.array([res["plane"][k] for k in ("z0", "theta_u", "theta_v")]),
                 virtual_xyz=res["virtual"].xyz, virtual_xy=res["virtual"].xy, query_obs=query_red[s].obs_names.to_numpy(str))
    gc.collect()

### Full-resolution scores

Each query is reloaded once more, for the expression of the evaluation genes on every bin. Every bin takes the transferred class and the transported expression of its point, as in the 2D notebook.

In [ ]:
for row in rows:
    s = row["query"]
    q = QUERIES[s]
    q["Y_full"] = cs.load_full_expression(QUERY_DS, path(QUERY_DS, s), eval_query, DROP_LABELS)
    res = results[s]
    vs = res["virtual"]
    ev = evaluate(res["P"], q, vs.labels(ST["labels"]), vs.interp(ST["Y"]))
    row.update({k: ev[k] for k in ("label_acc_full", "label_bal_acc_full", "coexpr_r_full", "coexpr_r_shuffled_full")})
    for kind, (sec, P2) in res["comparisons"].items():
        m = ST["section"] == sec
        ev2 = evaluate(P2, q, ST["labels"][m], ST["Y"][m])
        row.update({f"{kind}_{k}": ev2[k] for k in ("label_acc_full", "coexpr_r_full")})
    del q["Y_full"]
    gc.collect()

# 4. Summary

One row per query. The z₀ order of the queries is compared with the query embryo's file order: neighbouring sections of one embryo should land next to each other. The sign of z is arbitrary, so the comparison uses |ρ|. The rows are appended to `stack3d_summary.csv` with the run settings.

In [ ]:
summary = pd.DataFrame(rows)
if len(summary):
    show = ["query", "L_query", "z0", "z0_scan", "z0_sections", "theta_u", "theta_v", "coverage", "edge", "det", "nearest_section",
            "label_acc", "label_acc_full", "coexpr_r", "coexpr_r_full",
            "L_partner_section", "L_partner_label_acc_full", "L_partner_coexpr_r_full",
            "best_real_section_section", "best_real_section_label_acc_full", "best_real_section_coexpr_r_full"]
    print(summary[[c for c in show if c in summary]].round(3).to_string(index=False))
    if len(summary) > 2:
        rho = spearmanr(summary.z0, [query_names.index(s) for s in summary["query"]])[0]
        print(f"\nSpearman between z0 and the {emb_name(QUERY_EMB)} file order: {rho:+.2f} (|rho| {abs(rho):.2f})")
    print("implied order of the queries along the stack:", list(summary.sort_values("z0")["query"]))

meta = dict(date=datetime.now().isoformat(timespec="seconds"), direction=DIRECTION, stack=emb_name(STACK_EMB),
            stack_dataset=STACK_DS, query_embryo=emb_name(QUERY_EMB), method=METHOD, order_mode=ORDER,
            stack_order=";".join(stack_names), z_sections=";".join(f"{z:.4f}" for z in z_sec),
            max_obs_stack=MAX_OBS_STACK, max_obs_query=MAX_OBS_QUERY, subsample=SUBSAMPLE, agg=AGG,
            icp_lambda=ICP_LAMBDA, tau_stack=TAU_STACK, coarse_alpha=COARSE_ALPHA, fgw_alpha=FGW_ALPHA,
            tilt_max=TILT_MAX, n_z=N_Z, mirror_test=MIRROR_TEST, z_gaps="estimated" if Z_GAPS is None else "manual",
            virtual_features=VIRTUAL_FEATURES, z_select=Z_SELECT,
            ss_mae_scan=float(np.mean(np.abs(ss.z0_scan - ss.true_z))) if len(ss_rows) else np.nan,
            ss_mae_sections=float(np.mean(np.abs(ss.z0_sections - ss.true_z))) if len(ss_rows) else np.nan)
SUMMARY_CSV = SAVE_DIR / "stack3d_summary.csv"
out = summary.assign(**meta)
if SUMMARY_CSV.exists():
    out = pd.concat([pd.read_csv(SUMMARY_CSV), out], ignore_index=True)
out.to_csv(SUMMARY_CSV, index=False)
print("appended to", SUMMARY_CSV)

# 5. Both directions

Once both directions have run, each embryo has been a stack once and a set of queries once. Two orders can then be compared for each embryo:
- its stack order (file order, or inferred from its own sections);
- the order its sections take when they are placed into the other embryo's stack.

Agreement means the placement recovers the anatomy of the sections. This matters most for E12.5_E1, whose file numbers are probably not its physical order.

In [ ]:
allsum = pd.read_csv(SUMMARY_CSV)
for emb in (emb_name(HUMAN_EMBRYO), emb_name(MOUSE_EMBRYO)):
    f = SAVE_DIR / f"order_{emb}.json"
    placed = allsum[(allsum.query_embryo == emb) & (allsum.method == METHOD)]
    if not f.exists() or placed.empty:
        print(f"{emb}: run both directions first")
        continue
    placed = placed[placed.date == placed.date.max()]
    stack_order = json.load(open(f))["order"]
    common = [s for s in stack_order if s in set(placed["query"])]
    implied = list(placed.set_index("query").loc[common].sort_values("z0").index)
    rho = spearmanr([stack_order.index(s) for s in common], placed.set_index("query").loc[common, "z0"])[0] if len(common) > 2 else np.nan
    print(f"{emb}: stack order {stack_order}\n{'':>{len(emb) + 2}}placed in the other stack: {implied}  (|Spearman| {abs(rho):.2f})")